# 05 — Systematic Single-Image Clustering Experiments

Loops over **all 5 image types × 2 distance metrics × 2 k values = 20 combinations**
and records evaluation metrics for each.

| Image type | File |
|---|---|
| Coronal | `pca_cor.csv` |
| Sagittal GFC | `pca_gfc_sag.csv` |
| Transverse GFC | `pca_gfc_tra.csv` |
| Brain-masked transverse | `pca_masked_tra.csv` |
| Native sagittal | `pca_sbj_sag.csv` |

**Distances:** `euclidean`, `robust_mahalanobis`  
**k values:** 3, 4  
**Evaluation:** crosstab vs CDR · % healthy spread · Silhouette · ARI · NMI

## Step 1 — Install dependencies

In [1]:
import subprocess, sys

def run_pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet"] + list(args)
    r   = subprocess.run(cmd, capture_output=True, text=True)
    tag = args[-1]
    if r.returncode == 0:
        print(f"  OK    {tag}")
    else:
        print(f"  FAIL  {tag}")
        print(r.stderr[-600:] if r.stderr else "(no stderr)")
    return r.returncode

# --no-deps skips scikit-learn-extra (requires MSVC to compile on Windows)
run_pip("--no-deps", "db-robust-clust==0.1.10")
for pkg in ["kmedoids", "robust-mixed-dist", "openpyxl"]:
    run_pip(pkg)
print("Done.")

  OK    db-robust-clust==0.1.10
  OK    kmedoids
  OK    robust-mixed-dist
  OK    openpyxl
Done.


## Step 2 — Imports

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kmedoids
from db_robust_clust.models import SampleDistClustering
from robust_mixed_dist.mixed import (
    generalized_gower_dist_matrix,
    robust_mahalanobis_dist_matrix,
    simple_gower_dist_matrix,
    S_robust,
)
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
)

print("All imports successful.")

All imports successful.


## Shared setup — load CDR once

Clinical data is loaded once here and reused across all 20 experiment combinations.

In [3]:
DATA_DIR   = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
NB_DIR     = os.path.join(DATA_DIR, "notebooks")
NB_PADDED  = os.path.join(NB_DIR, "new_results_with_padding")

df_clin = pd.read_excel(
    os.path.join(NB_DIR, "oasis_cross-sectional-5708aa0a98d82080.xlsx")
)
if "ID" in df_clin.columns:
    df_clin = df_clin.rename(columns={"ID": "patient_id"})

df_clin["patient_id"] = df_clin["patient_id"].str.replace(r"_MR\d+$", "", regex=True)
df_clin = (
    df_clin
    .sort_values("CDR", na_position="last")
    .drop_duplicates(subset="patient_id", keep="first")
    .reset_index(drop=True)
)
cdr_lookup = df_clin.set_index("patient_id")["CDR"]

print(f"Clinical table : {df_clin.shape}")
print(f"CDR non-null   : {int(cdr_lookup.notnull().sum())} patients")

Clinical table : (416, 12)
CDR non-null   : 235 patients


## Main experiment loop

5 images × 2 metrics × 2 k values = **20 combinations**.

For `robust_mahalanobis` the distance matrix is precomputed once per image type and
reused across both k values to avoid redundant computation.

In [4]:
PCA_FILES = [
    "pca_cor.csv",
    "pca_gfc_sag.csv",
    "pca_gfc_tra.csv",
    "pca_masked_tra.csv",
    "pca_sbj_sag.csv",
]
METRICS  = ["euclidean", "robust_mahalanobis"]
K_VALUES = [3, 4]

all_results = []

for pca_file in PCA_FILES:
    image_type  = pca_file.replace("pca_", "").replace(".csv", "")
    df_pca      = pd.read_csv(os.path.join(NB_DIR, pca_file))
    patient_ids = df_pca["patient_id"].values
    X           = df_pca.drop(columns=["patient_id"]).to_numpy(dtype=np.float64)

    # CDR aligned to this image's patient order
    cdr_s    = cdr_lookup.reindex(patient_ids)
    mask_cdr = cdr_s.notnull().values
    cdr_int  = (cdr_s.values[mask_cdr] * 2).astype(int)  # 0→0, 0.5→1, 1→2, 2→4

    print(f"\n{'='*62}")
    print(f"  {image_type}  |  X={X.shape}  |  CDR patients: {mask_cdr.sum()}")
    print(f"{'='*62}")

    for metric in METRICS:

        # Precompute robust Mahalanobis D once per (image, metric) — reused for k=3 and k=4
        if metric == "robust_mahalanobis":
            print(f"  [{metric}] computing distance matrix ...", end=" ", flush=True)
            S_est     = S_robust(X, method="trimmed", alpha=0.05)
            D_precomp = robust_mahalanobis_dist_matrix(X, S_est)
            print("done")

        for k in K_VALUES:
            base = kmedoids.KMedoids(
                n_clusters=k, metric="precomputed", method="pam", random_state=42
            )
            model = SampleDistClustering(
                clustering_method=base,
                metric=metric,
                frac_sample_size=1.0,
                random_state=42,
                p1=110,
            )
            model.fit(X)
            labels = np.array(model.labels_, dtype=int)

            # ── % healthy per cluster (CDR = 0.0) ────────────────────────────
            df_eval  = pd.DataFrame({"cluster": labels, "CDR": cdr_s.values})
            df_eval  = df_eval[df_eval["CDR"].notnull()].copy()
            pct_tab  = pd.crosstab(
                df_eval["cluster"], df_eval["CDR"], normalize="index"
            ).mul(100).round(1)
            pct_healthy = pct_tab[0.0] if 0.0 in pct_tab.columns \
                          else pd.Series(0.0, index=pct_tab.index)
            spread = float(pct_healthy.max() - pct_healthy.min())

            # ── Silhouette ────────────────────────────────────────────────────
            if metric == "euclidean":
                sil = silhouette_score(X, labels, metric="euclidean")
            else:
                sil = silhouette_score(D_precomp, labels, metric="precomputed")

            # ── ARI / NMI ─────────────────────────────────────────────────────
            ari = adjusted_rand_score(cdr_int, labels[mask_cdr])
            nmi = normalized_mutual_info_score(cdr_int, labels[mask_cdr])

            sizes = {c: int((labels == c).sum()) for c in sorted(set(labels))}
            print(f"\n  {metric}  k={k}")
            print(f"    sizes      : {sizes}")
            print(f"    %healthy   : { {c: round(v,1) for c,v in pct_healthy.items()} }")
            print(f"    spread={spread:.1f}%  sil={sil:.4f}  ARI={ari:.4f}  NMI={nmi:.4f}")

            all_results.append({
                "image_type":         image_type,
                "metric":             metric,
                "k":                  k,
                "silhouette":         round(sil, 4),
                "ari":                round(ari, 4),
                "nmi":                round(nmi, 4),
                "spread_pct_healthy": round(spread, 2),
            })

print("\n\nAll 20 experiments complete.")


  cor  |  X=(416, 110)  |  CDR patients: 235

  euclidean  k=3
    sizes      : {np.int64(0): 137, np.int64(1): 152, np.int64(2): 127}
    %healthy   : {0: 66.0, 1: 64.6, 2: 47.6}
    spread=18.4%  sil=0.0223  ARI=-0.0079  NMI=0.0221

  euclidean  k=4
    sizes      : {np.int64(0): 106, np.int64(1): 106, np.int64(2): 108, np.int64(3): 96}
    %healthy   : {0: 65.1, 1: 57.8, 2: 46.9, 3: 78.1}
    spread=31.2%  sil=0.0079  ARI=-0.0244  NMI=0.0368
  [robust_mahalanobis] computing distance matrix ... done

  robust_mahalanobis  k=3
    sizes      : {np.int64(0): 174, np.int64(1): 135, np.int64(2): 107}
    %healthy   : {0: 52.5, 1: 60.0, 2: 62.5}
    spread=10.0%  sil=0.0005  ARI=-0.0135  NMI=0.0127

  robust_mahalanobis  k=4
    sizes      : {np.int64(0): 155, np.int64(1): 111, np.int64(2): 89, np.int64(3): 61}
    %healthy   : {0: 57.0, 1: 64.2, 2: 65.2, 3: 36.1}
    spread=29.1%  sil=0.0011  ARI=0.0094  NMI=0.0289

  gfc_sag  |  X=(416, 110)  |  CDR patients: 235

  euclidean  k=3
    

## Results — summary table and export

In [5]:
df_results = pd.DataFrame(all_results)

print("All results (5 images × 2 metrics × 2 k = 20 rows):")
print(df_results.to_string(index=False))

out_path = os.path.join(NB_PADDED, "results_single_image.csv")
df_results.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

All results (5 images × 2 metrics × 2 k = 20 rows):
image_type             metric  k  silhouette     ari    nmi  spread_pct_healthy
       cor          euclidean  3      0.0223 -0.0079 0.0221                18.4
       cor          euclidean  4      0.0079 -0.0244 0.0368                31.2
       cor robust_mahalanobis  3      0.0005 -0.0135 0.0127                10.0
       cor robust_mahalanobis  4      0.0011  0.0094 0.0289                29.1
   gfc_sag          euclidean  3      0.0235  0.0170 0.0189                18.8
   gfc_sag          euclidean  4      0.0210  0.0009 0.0313                19.2
   gfc_sag robust_mahalanobis  3      0.0029 -0.0051 0.0070                 0.9
   gfc_sag robust_mahalanobis  4      0.0023 -0.0014 0.0158                 8.0
   gfc_tra          euclidean  3      0.0009 -0.0121 0.0442                33.0
   gfc_tra          euclidean  4      0.0104 -0.0031 0.0485                34.7
   gfc_tra robust_mahalanobis  3      0.0014 -0.0032 0.0124         